In [2]:
from pathlib import Path
import pandas as pd

# Define paths
raw_path = Path("./ieee_fraud/bronze/raw")
out_path = Path("./ieee_fraud/silver")
out_path.mkdir(parents=True, exist_ok=True)


In [3]:


# Map PySpark types to Pandas-compatible type equivalents
dtype_mapping = {"BIGINT": "Int64", "DOUBLE": "float64", "INT": "Int32", "STRING": "object"}

# Explicit numeric identity measurements (matching your logic)
numeric_identity = {
    f"id_{i:02d}" for i in (list(range(1, 12)) + [13, 14] + list(range(17, 23)) + [24, 25, 26, 32])
}


# Define the intended type of each feature (equivalent to your PySpark function)
def get_intended_type(table_name, col_name):
  if col_name in {"TransactionID", "TransactionDT"}:
    return "BIGINT"
  if col_name == "TransactionAmt":
    return "DOUBLE"
  if col_name == "isFraud":
    return "INT"
  if col_name in numeric_identity:
    return "DOUBLE"

  if col_name.startswith(("C", "D", "V")):
    prefix, suffix = col_name[0], col_name[1:]
    if suffix.isdigit():
      num = int(suffix)
      if (
          (prefix == "C" and 1 <= num <= 14)
          or (prefix == "D" and 1 <= num <= 15)
          or (prefix == "V" and 1 <= num <= 339)
      ):
        return "DOUBLE"

  if col_name in {"dist1", "dist2"}:
    return "DOUBLE"

  return "STRING"


# Source tables mapping for files
tables = {
    "ieee_train_transaction": "train_transaction.csv",
    "ieee_test_transaction": "test_transaction.csv",
    "ieee_train_identity": "train_identity.csv",
    "ieee_test_identity": "test_identity.csv",
}

manifest = []
chunk_size = 100000  # Adjust based on memory availability

for table_name, file_name in tables.items():
  file_path = raw_path / file_name
  print(f"Processing chunked scrub for {table_name}...")

  # Read the first chunk to inspect columns and establish schema manifest
  first_chunk = pd.read_csv(file_path, nrows=100)

  # Standardize identity column names if applicable (fixing hyphenated fields like id-24 -> id_24)
  if "identity" in table_name:
    first_chunk.columns = [c.replace("-", "_") for c in first_chunk.columns]

  # Build manifest rows equivalent to your Spark schema script
  for ordinal, field_name in enumerate(first_chunk.columns, start=1):
    sql_type = get_intended_type(table_name, field_name)
    manifest.append((table_name, ordinal, field_name, sql_type, True))

  # Process the full file in chunks to prevent OOM kernel deaths
  temp_chunks_dir = out_path / f"temp_{table_name}"
  temp_chunks_dir.mkdir(parents=True, exist_ok=True)

  chunk_idx = 0
  for chunk in pd.read_csv(file_path, chunksize=chunk_size):
    if "identity" in table_name:
      chunk.columns = [c.replace("-", "_") for c in chunk.columns]

    # Apply type conversions chunk-by-chunk based on intended types
    for col in chunk.columns:
      sql_type = get_intended_type(table_name, col)
      pandas_type = dtype_mapping.get(sql_type, "object")
      try:
        chunk[col] = chunk[col].astype(pandas_type)
      except Exception:
        # Fallback if nulls prevent strict casting initially
        pass

    chunk_file = temp_chunks_dir / f"chunk_{chunk_idx}.parquet"
    chunk.to_parquet(chunk_file, index=False)
    chunk_idx += 1

  # Combine chunks back into a single unified silver Parquet file
  print(f"Concatenating chunks into final silver table for {table_name}...")
  all_chunk_files = sorted(temp_chunks_dir.glob("chunk_*.parquet"))
  combined_df = pd.concat([pd.read_parquet(f) for f in all_chunk_files], ignore_index=True)

  final_output_file = out_path / f"{table_name}.parquet"
  combined_df.to_parquet(final_output_file, index=False)
  print(f"Saved: {final_output_file}")

# Create the equivalent schema documentation manifest DataFrame
schema_df = pd.DataFrame(
    manifest, columns=["table_name", "ordinal_position", "column_name", "intended_type", "nullable"]
)

# Display the resulting schema manifest cleanly in your notebook
schema_df.sort_values(by=["table_name", "ordinal_position"], inplace=True)
display(schema_df)

Processing chunked scrub for ieee_train_transaction...
Concatenating chunks into final silver table for ieee_train_transaction...
Saved: ieee_fraud/silver/ieee_train_transaction.parquet
Processing chunked scrub for ieee_test_transaction...
Concatenating chunks into final silver table for ieee_test_transaction...
Saved: ieee_fraud/silver/ieee_test_transaction.parquet
Processing chunked scrub for ieee_train_identity...
Concatenating chunks into final silver table for ieee_train_identity...
Saved: ieee_fraud/silver/ieee_train_identity.parquet
Processing chunked scrub for ieee_test_identity...
Concatenating chunks into final silver table for ieee_test_identity...
Saved: ieee_fraud/silver/ieee_test_identity.parquet


,table_name,ordinal_position,column_name,intended_type,nullable
828,ieee_test_identity,1,TransactionID,BIGINT,True
829,ieee_test_identity,2,id_01,DOUBLE,True
830,ieee_test_identity,3,id_02,DOUBLE,True
831,ieee_test_identity,4,id_03,DOUBLE,True
832,ieee_test_identity,5,id_04,DOUBLE,True
...,...,...,...,...,...
389,ieee_train_transaction,390,V335,DOUBLE,True
390,ieee_train_transaction,391,V336,DOUBLE,True
391,ieee_train_transaction,392,V337,DOUBLE,True
392,ieee_train_transaction,393,V338,DOUBLE,True


In [1]:
import pyarrow.parquet as pq
import pandas as pd
from pathlib import Path

# Paths configuration
out_path = Path("./ieee_fraud/silver")
train_trans_file = out_path / "ieee_train_transaction.parquet"
train_ident_file = out_path / "ieee_train_identity.parquet"
merged_out_dir = out_path / "temp_merged_chunks"
merged_out_dir.mkdir(parents=True, exist_ok=True)

print("Loading identity table into memory (identity table is compact)...")
# The identity table is much smaller, so it fits safely into RAM
train_ident = pd.read_parquet(train_ident_file)

# Set TransactionID as index for lightning-fast lookups during iteration
if "TransactionID" in train_ident.columns:
    train_ident = train_ident.set_index("TransactionID")

batch_size = 50000
chunk_idx = 0

print("Starting iterative chunked join for training data...")
parquet_file = pq.ParquetFile(str(train_trans_file))

for batch in parquet_file.iter_batches(batch_size=batch_size):
    # Convert PyArrow batch to a manageable Pandas chunk
    trans_chunk = batch.to_pandas()
    
    # Perform a left join against the pre-loaded identity dataframe in memory
    # Using index-based joining keeps it extremely memory-efficient
    merged_chunk = trans_chunk.merge(
        train_ident, 
        left_on="TransactionID", 
        right_index=True, 
        how="left"
    )
    
    # Flush the merged chunk immediately to disk
    chunk_file = merged_out_dir / f"merged_chunk_{chunk_idx:04d}.parquet"
    merged_chunk.to_parquet(chunk_file, index=False)
    
    print(f"Processed and flushed merged training chunk {chunk_idx} ({len(merged_chunk)} rows).")
    chunk_idx += 1

print("Concatenating final unified training parquet from disk chunks...")
all_merged_files = sorted(merged_out_dir.glob("merged_chunk_*.parquet"))

# Use pyarrow dataset / table concatenation to write the final file without loading all into RAM
import pyarrow as pa
import pyarrow.parquet as parquet

tables = [parquet.read_table(f) for f in all_merged_files]
combined_table = pa.concat_tables(tables)

final_merged_file = out_path / "ieee_train_merged.parquet"
pq.write_table(combined_table, final_merged_file)
print(f"Successfully saved unified training table to: {final_merged_file}")

Loading identity table into memory (identity table is compact)...
Starting iterative chunked join for training data...
Processed and flushed merged training chunk 0 (50000 rows).
Processed and flushed merged training chunk 1 (50000 rows).
Processed and flushed merged training chunk 2 (50000 rows).
Processed and flushed merged training chunk 3 (50000 rows).
Processed and flushed merged training chunk 4 (50000 rows).
Processed and flushed merged training chunk 5 (50000 rows).
Processed and flushed merged training chunk 6 (50000 rows).
Processed and flushed merged training chunk 7 (50000 rows).
Processed and flushed merged training chunk 8 (50000 rows).
Processed and flushed merged training chunk 9 (50000 rows).
Processed and flushed merged training chunk 10 (50000 rows).
Processed and flushed merged training chunk 11 (40540 rows).
Concatenating final unified training parquet from disk chunks...
Successfully saved unified training table to: ieee_fraud/silver/ieee_train_merged.parquet


In [1]:
import pyarrow.parquet as pq
from pathlib import Path

# Paths setup
silver_path = Path("./ieee_fraud/silver")
merged_file = silver_path / "ieee_train_merged.parquet"

# 1. Verify exact row count using Parquet metadata (zero memory overhead)
parquet_file = pq.ParquetFile(str(merged_file))
total_rows = parquet_file.metadata.num_rows

print(f"Unified Table: {merged_file.name}")
print(f"Total Row Count: {total_rows:,}")

expected_rows = 590540
if total_rows == expected_rows:
    print("✔ Row count matches the expected IEEE-CIS training size (590,540 rows).")
else:
    print(f"⚠ Note: Expected {expected_rows:,} rows, but found {total_rows:,}.")

# 2. Define chronological split boundaries (70% Train, 15% Val, 15% Test)
train_end = int(total_rows * 0.70)
val_end = int(total_rows * 0.85)

print("\n--- Split Plan ---")
print(f"Train Split       : Rows 0 to {train_end:,} (~70%)")
print(f"Validation Split  : Rows {train_end:,} to {val_end:,} (~15%)")
print(f"Test Split        : Rows {val_end:,} to {total_rows:,} (~15%)")

# 3. Read table as a PyArrow Table (columnar format avoids Pandas deserialization OOM)
table = parquet_file.read()

# Slice the PyArrow table cleanly
train_table = table.slice(0, train_end)
val_table = table.slice(train_end, val_end - train_end)
test_table = table.slice(val_end, total_rows - val_end)

# 4. Save the splits to your silver directory
train_out = silver_path / "ieee_train_split_merged.parquet"
val_out = silver_path / "ieee_validation_split_merged.parquet"
test_out = silver_path / "ieee_test_split_merged.parquet"

pq.write_table(train_table, train_out)
pq.write_table(val_table, val_out)
pq.write_table(test_table, test_out)

print("\n--- Successfully Saved Splits ---")
print(f"Saved: {train_out.name} ({len(train_table):,} rows)")
print(f"Saved: {val_out.name} ({len(val_table):,} rows)")
print(f"Saved: {test_out.name} ({len(test_table):,} rows)")

Unified Table: ieee_train_merged.parquet
Total Row Count: 590,540
✔ Row count matches the expected IEEE-CIS training size (590,540 rows).

--- Split Plan ---
Train Split       : Rows 0 to 413,378 (~70%)
Validation Split  : Rows 413,378 to 501,959 (~15%)
Test Split        : Rows 501,959 to 590,540 (~15%)

--- Successfully Saved Splits ---
Saved: ieee_train_split_merged.parquet (413,378 rows)
Saved: ieee_validation_split_merged.parquet (88,581 rows)
Saved: ieee_test_split_merged.parquet (88,581 rows)


In [ ]:

# Validate unique primary keys
assert (
    train_transaction["TransactionID"].nunique() == len(train_transaction)
), "Duplicate TransactionIDs found in train_transaction!"
assert (
    test_transaction["TransactionID"].nunique() == len(test_transaction)
), "Duplicate TransactionIDs found in test_transaction!"
print("  - Primary key integrity check passed: Zero duplicate TransactionIDs.")




In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully!")
print("Model device:", model.device)

[transformers] Disabling PyTorch because PyTorch >= 2.6 is required but found 2.5.1+cu121
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

ImportError: 
AutoModelForCausalLM requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.
